# Embedding层和 Linear层

- 在PyTorch中，嵌入层（Embedding layers）实现了执行矩阵乘法的线性层的相同功能；我们使用嵌入层的原因是为了提高计算效率。
- 我们将逐步使用PyTorch中的代码示例来查看这种关系。

In [2]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.14.1+cpu


## Using nn.Embedding

In [3]:
# 假设我们有一个句子，分词后包含3个token，映射为词id后
idx = torch.tensor([2, 3, 1])

# 词嵌入矩阵的行数 = 词表大小，列数 = 嵌入维度
# 为了可重复性，我们使用随机种子，
# 因为嵌入层的权重是用小的随机值初始化的
torch.manual_seed(123)
embedding = torch.nn.Embedding(5, 10)

查看嵌入权重数据情况

In [4]:
embedding.weight

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.3035, -0.5880,  0.3486,  0.6603, -0.2196, -0.3792,
          0.7671, -1.1925],
        [ 0.6984, -1.4097,  0.1794,  1.8951,  0.4954,  0.2692, -0.0770, -1.0205,
         -0.1690,  0.9178],
        [ 1.5810,  1.3010,  1.2753, -0.2010,  0.4965, -1.5723,  0.9666, -1.1481,
         -1.1589,  0.3255],
        [-0.6315, -2.8400, -1.3250,  0.1784, -1.4779,  1.1331, -1.2203,  1.3139,
          1.0533,  0.1388],
        [-0.2044, -2.2685, -0.2808,  0.7697, -0.6596, -0.7979,  0.1838,  0.2293,
          0.6177, -0.2876]], requires_grad=True)

- 使用嵌入层来获取具有ID 1的训练样本的向量表示

In [5]:
embedding(torch.tensor([1]))

tensor([[ 0.6984, -1.4097,  0.1794,  1.8951,  0.4954,  0.2692, -0.0770, -1.0205,
         -0.1690,  0.9178]], grad_fn=<EmbeddingBackward0>)

- 现在，让我们将之前定义的所有训练样本转换：

In [7]:
embed = embedding(idx)
print(embed.shape)

torch.Size([3, 10])


## 使用 nn.Linear

- 接下来，我们将使用One-Hot编码，与embedding 层一样，在 `nn.Linear` 层进行操作
- 首先，我们将标记ID转换为One-Hot表示：

In [ ]:
onehot = torch.nn.functional.one_hot(idx)
print(onehot)

tensor([[0, 0, 1, 0],
        [0, 0, 0, 1],
        [0, 1, 0, 0]])

- 接下来，我们使用矩阵乘法$X W^\top$ 来初始化一个Linear层

In [9]:
torch.manual_seed(123)
# 初始化一个Linear层，该层的权重矩阵是由 num_idx（输入维度）到 out_dim（输出维度）的一个线性层，而且没有偏置项
linear = torch.nn.Linear(5, 10, bias=False)
print(linear.weight)

Parameter containing:
tensor([[-0.1824,  0.0148, -0.2221,  0.1687, -0.3811],
        [ 0.3278, -0.3251, -0.3556, -0.2826,  0.2025],
        [-0.1652,  0.1674, -0.3796, -0.2713, -0.1642],
        [-0.0879, -0.3412,  0.2928, -0.1055,  0.1436],
        [ 0.3162,  0.0833,  0.1223,  0.4317, -0.2017],
        [ 0.1417, -0.1990,  0.3196,  0.3572, -0.4123],
        [ 0.3818,  0.2136,  0.1949,  0.1841,  0.3718],
        [-0.0590, -0.3782, -0.1283, -0.3150,  0.0296],
        [-0.0835, -0.2399, -0.0407,  0.4237, -0.0353],
        [ 0.0142, -0.0697,  0.0703,  0.3985,  0.2735]], requires_grad=True)


- 请注意，PyTorch中的`linear`层也是用小的随机权重进行初始化的。为了与上面的 `Embedding` 层进行直接比较，我们必须使用相同的小随机权重，这就是我们在这里重新分配它们的原因：

In [17]:
# linear 层的权重就被重新赋值为与 embedding 层相同的小随机权重，以确保它们具有相同的初始化。这是为了使它们在后续操作中可以进行直接比较。
linear.weight = torch.nn.Parameter(embedding.weight.T.detach())

- 现在，我们可以使用线性层处理输入的One-Hot编码表示：

In [10]:
linear(onehot.float())

RuntimeError: mat1 and mat2 shapes cannot be multiplied (3x4 and 5x10)

正如我们所看到的，这与我们使用嵌入层时得到的结果完全相同：